### Boole's rule

In [1]:
import numpy as np
import sympy as smp
from scipy.integrate import quad, simpson, trapezoid
from sympy.polys.polyfuncs import interpolating_poly  # Symbolic interpolation

In [2]:
x = smp.symbols('x', real = True) # Free variable
h, L, N = smp.symbols('h L N', real = True, constant = True, positive = True) # h step size, L length of integration interval (b - a)
m = smp.symbols('m', integer = True, positive = True)
y = smp.Function('y', real = True)(x) # Function to integrate y(x)

# Take 5 points equally spaced
x_points = [- 2 * h, - h, 0, h, 2 * h]
y_points = [
    y.subs(x, -2 * h),
    y.subs(x, -h),
    y.subs(x, 0),
    y.subs(x, h),
    y.subs(x, 2 * h)
    ]

In [3]:
pol = interpolating_poly( # Interpolate with a 4th order polynomial
    n = 5,
    x = x,
    X = x_points,
    Y = y_points
    )

pol

x*(-2*h + x)*(-h + x)*(h + x)*y(-2*h)/(24*h**4) - x*(-2*h + x)*(-h + x)*(2*h + x)*y(-h)/(6*h**4) - x*(-2*h + x)*(h + x)*(2*h + x)*y(h)/(6*h**4) + x*(-h + x)*(h + x)*(2*h + x)*y(2*h)/(24*h**4) + (-2*h + x)*(-h + x)*(h + x)*(2*h + x)*y(0)/(4*h**4)

In [4]:
I = smp.integrate(pol, (x, -2 * h, 2 * h)) # Integrate the polynomial. Boole's rule found for one panel
I = I.factor()
I

2*h*(12*y(0) + 7*y(-2*h) + 32*y(-h) + 32*y(h) + 7*y(2*h))/45

In [5]:
err = smp.Abs(smp.Integral(y, (x, -2 * h, 2 * h)) - I) # Definition of absolute error
err

Abs(2*h*(12*y(0) + 7*y(-2*h) + 32*y(-h) + 32*y(h) + 7*y(2*h))/45 - Integral(y(x), (x, -2*h, 2*h)))

In [6]:
y_exp = smp.series(y, x, 0, 8).removeO() # Taylor expansion of y(x) around 0
y_exp

x**7*Subs(Derivative(y(xi), (xi, 7)), xi, 0)/5040 + x**6*Subs(Derivative(y(xi), (xi, 6)), xi, 0)/720 + x**5*Subs(Derivative(y(xi), (xi, 5)), xi, 0)/120 + x**4*Subs(Derivative(y(xi), (xi, 4)), xi, 0)/24 + x**3*Subs(Derivative(y(xi), (xi, 3)), xi, 0)/6 + x**2*Subs(Derivative(y(xi), (xi, 2)), xi, 0)/2 + x*Subs(Derivative(y(xi), xi), xi, 0) + y(0)

In [7]:
# Taylor expansion of y(x) around 0 evaluated at interesting points
y_minus_2h_exp = y_exp.subs(x, -2 * h)
y_minus_h_exp = y_exp.subs(x, -h)
y_plus_h_exp = y_exp.subs(x, h)
y_plus_2h_exp = y_exp.subs(x, 2 * h)

In [8]:
subs = {
    y: y_exp,
    y.subs(x, -2 * h): y_minus_2h_exp,
    y.subs(x, -h): y_minus_h_exp,
    y.subs(x, h): y_plus_h_exp,
    y.subs(x, 2 * h): y_plus_2h_exp,
}

err = err.subs(subs).doit().expand() # Computing the error, absolute value
err

8*h**7*Abs(Subs(Derivative(y(xi), (xi, 6)), xi, 0))/945

In [9]:
total_err = err * L / (4 * h) # Total error committed for the whole integral int_{a}^{b} y(x) dx
total_err

2*L*h**6*Abs(Subs(Derivative(y(xi), (xi, 6)), xi, 0))/945

In [10]:
total_err = total_err.subs(h, L / (N - 1)) # In terms of N, h = L / (N - 1)
total_err

2*L**7*Abs(Subs(Derivative(y(xi), (xi, 6)), xi, 0))/(945*(N - 1)**6)

In [11]:
total_err = total_err.subs(N, 4 * m + 1) # In terms of m, N = 4 * m + 1 (composite rule)
total_err

L**7*Abs(Subs(Derivative(y(xi), (xi, 6)), xi, 0))/(1935360*m**6)

In [12]:
def boole(f, a, b, m): # Composite Boole's rule

    N = 4 * m + 1 # N must be a natural number of the form 4 * m + 1, N = 5, 9, 13, 17, ... and so on

    x = np.linspace(a, b, N, endpoint = True)
    y = f(x)

    # Boole weights
    w = np.empty(N, dtype = float)
    w[0] = 7.0
    w[-1] = 7.0
    w[1::4] = 32.0
    w[2::4] = 12.0
    w[3::4] = 32.0
    w[4:-1:4] = 14.0

    return (2.0 * ((b - a) / (N - 1)) / 45.0) * w.dot(y) # Boole's rule via dot product

Let's compute:

$$
    \int_{0}^{3} \sqrt{\left| \frac{x - 3}{x + 3} \right|} \, dx
$$

In [13]:
def f(x): return np.sqrt(abs((x - 3.0) / (x + 3.0))) # Integrand function
a, b = 0.0, 3.0 # Extremes

In [14]:
boole(f, a, b, 1), boole(f, a, b, 2), boole(f, a, b, 3) # N = 5, 9, 13

(np.float64(1.6936719927432746),
 np.float64(1.7057341271617807),
 np.float64(1.7087612468495008))

The absolute error (truncation error) committed when computing $I(m)$ can be estimated as:

$$
    E(m) = \frac{(m + 1)^6}{\left( 2 m + 1 \right) \left( m^2 + m + 1 \right) \left( 3 m^2 + 3 m + 1 \right)} \, \left| I(m + 1) - I(m) \right|
$$

Derivation available in the *PDF* folder

In [ ]:
def boole_err(f, a, b, m): # Applies Richardson method to estimate the absolute error of I(m)

    u = np.finfo(np.float64).eps / 2.0 # Roundoff unit error

    I_m0 = boole(f, a, b, m)
    I_m1 = boole(f, a, b, m + 1)

    diff = np.abs(I_m1 - I_m0)
    err_I_m = (m + 1.0) ** 6 / ((2.0 * m + 1.0) * (m ** 2 + m + 1.0) * (3.0 * m ** 2 + 3.0 * m + 1.0)) * diff # Error formula
    cc_rel = (np.abs(I_m0) + np.abs(I_m1)) / diff * u # Catastrophic cancellation of the difference |I(m + 1) - I(m)|

    # Integral value, absolute error estimated, relative error of the difference |I(m + 1) - I(m)| due catastrophic cancellation
    return (I_m0, err_I_m, cc_rel)

In [16]:
Is = np.array([boole_err(f, a, b, m) for m in range(1, 30 + 1, 1)]) # Computing some values for m = 1, 2, 3, ..., 30
Is

array([[1.69367199e+00, 1.22535969e-02, 3.12888152e-14],
       [1.70573413e+00, 3.31845151e-03, 1.25229650e-13],
       [1.70876125e+00, 1.54472152e-03, 2.98916288e-13],
       [1.71003104e+00, 9.08074346e-04, 5.66808396e-13],
       [1.71070107e+00, 6.06696470e-04, 9.41468583e-13],
       [1.71110458e+00, 4.39068143e-04, 1.43413880e-12],
       [1.71136953e+00, 3.35478986e-04, 2.05508252e-12],
       [1.71155445e+00, 2.66549631e-04, 2.81380202e-12],
       [1.71168952e+00, 2.18105198e-04, 3.71918585e-12],
       [1.71179171e+00, 1.82597251e-04, 4.77961419e-12],
       [1.71187124e+00, 1.55691827e-04, 6.00303713e-12],
       [1.71193456e+00, 1.34748473e-04, 7.39703478e-12],
       [1.71198595e+00, 1.18079462e-04, 8.96886453e-12],
       [1.71202833e+00, 1.04562486e-04, 1.07254990e-11],
       [1.71206378e+00, 9.34260315e-05, 1.26736574e-11],
       [1.71209377e+00, 8.41245333e-05, 1.48198308e-11],
       [1.71211943e+00, 7.62627034e-05, 1.71703043e-11],
       [1.71214157e+00, 6.95480

In [17]:
Is[-1, 0], Is[-1, 1], Is[-1, 2] # m = 30 results

(np.float64(1.7122739714466202),
 np.float64(3.090136417617176e-05),
 np.float64(6.889176929593528e-11))

With $m = 30$ (or $N = 121$), composite Boole's rule yields:

$$
    \int_{0}^{3} \, \sqrt{\left| \frac{x - 3}{x + 3} \right|} \, dx = 1.71227 \pm 0.00003
$$

Let's compare with other methods

In [18]:
real = quad(f, a, b)[0] # Gaussian-Legendre quadrature
print(f'{real:.5f}')

1.71239


In [19]:
# x and y points for m = 121
x = np.linspace(a, b, 121, endpoint = True)
y = f(x)

In [20]:
I_trap = trapezoid( # Trapezoid rule
    dx = x[1],
    y = y
)

I_trap

np.float64(1.712070780891795)

In [21]:
I_simp = simpson( # Simpson's rule
    dx = x[1],
    y = y
)

I_simp

np.float64(1.7122580166081653)

In [22]:
# Errors
abs(real - Is[- 1, 0]), abs(real - I_simp), abs(real - I_trap) # for m = 30, Boole is slightly better

(np.float64(0.00011500893806948298),
 np.float64(0.00013096377652432878),
 np.float64(0.0003181994928946885))